# exp98 — contacts-v1 train-set rollouts: interactive explorer

**1,000,000 rollouts** (1000 training targets × 1000 each) from the tuned contacts-v1 1.5B model (eval loss 2.7566), for [Open-Athena/MarinFold#98](https://github.com/Open-Athena/MarinFold/issues/98).

Each rollout is a sampled contacts-v1 contact set, scored (precision/recall/F1, all sep≥6) vs the document's ground-truth contacts. We keep every rollout's metrics and save the best-recall & best-F1 rollout per target verbatim.

**No login needed** — the data is in the *public* `open-athena/MarinFold` HF bucket. Just run the cells top to bottom.


In [ ]:
!pip -q install -U 'huggingface_hub>=1.5' pandas pyarrow matplotlib ipywidgets  # bucket fs needs hf_hub>=1.5

In [ ]:
import json, numpy as np, pandas as pd, matplotlib.pyplot as plt
from huggingface_hub import HfFileSystem

# Anonymous (token=False) read of the public bucket — no auth.
fs = HfFileSystem(token=False)
BASE = "buckets/open-athena/MarinFold/data/contacts-v1-train-rollouts-exp98"
def rp(name, **kw):
    with fs.open(f'{BASE}/{name}', 'rb') as f:
        return pd.read_parquet(f, **kw)

## Per-target summary (1000 targets)


In [ ]:
t = rp('per_target_summary.parquet')
print(len(t), 'targets')
t[['entry_id','L','n_gt','best_recall','best_f1','mean_rec','mean_gen_tokens','tokens_per_s']].describe().round(3)

## Overview — best-of-1000 accuracy (all sep≥6)


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for a, col in zip(ax, ['best_recall','best_f1']):
    a.hist(t[col], bins=30, color='steelblue', edgecolor='white')
    a.axvline(t[col].mean(), color='crimson', ls='--', label=f'mean {t[col].mean():.2f}')
    a.set_xlabel(col + ' (best of 1000)'); a.set_ylabel('# targets'); a.legend()
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].scatter(t.L, t.best_f1, s=8, alpha=.4, label='best F1')
ax[0].scatter(t.L, t.best_recall, s=8, alpha=.4, label='best recall')
ax[0].set_xlabel('L'); ax[0].set_ylabel('best of 1000'); ax[0].legend(); ax[0].set_title('accuracy vs length')
sc = ax[1].scatter(t.n_gt, t.best_f1, c=t.L, s=8, alpha=.5, cmap='viridis')
fig.colorbar(sc, ax=ax[1], label='L'); ax[1].set_xlabel('# GT contacts'); ax[1].set_ylabel('best F1'); ax[1].set_title('best F1 vs contact count')
plt.tight_layout(); plt.show()

## Best rollouts (the saved documents)
`best_rollouts.parquet` has, per target, the best-recall and best-F1 rollout saved verbatim as a complete contacts-v1 document (`*_document`) plus parsed `*_pred_contacts`.


In [ ]:
best = rp('best_rollouts.parquet')
best[['entry_id','L','n_gt','best_f1_f1','best_f1_recall','best_recall_recall']].sort_values('best_f1_f1', ascending=False).head(10)

## Drill into one target
`rollout_metrics_all.parquet` (1M rows) holds every rollout's per-band metrics, keyed by `entry_id`+`r`. We load it once (≈30 MB) and slice per target.


In [ ]:
ALL = rp('rollout_metrics_all.parquet')
print('total rollouts:', len(ALL))

def explore(entry):
    m = ALL[ALL.entry_id == entry]
    b = best[best.entry_id == entry].iloc[0]
    print(f'{entry}  L={int(b.L)}  n_gt={int(b.n_gt)}  ({len(m)} rollouts)')
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].scatter(m.all_rec, m.all_prec, s=6, alpha=.25)
    ax[0].set_xlabel('recall'); ax[0].set_ylabel('precision'); ax[0].set_title('per-rollout (all sep>=6)')
    ax[1].hist(m.all_f1, bins=30, color='steelblue', edgecolor='white')
    ax[1].set_xlabel('F1'); ax[1].set_ylabel('# rollouts'); ax[1].set_title('F1 distribution')
    plt.tight_layout(); plt.show()
    print(f"best F1   : F1={b.best_f1_f1:.3f} prec={b.best_f1_precision:.3f} rec={b.best_f1_recall:.3f}")
    print(f"best recall: rec={b.best_recall_recall:.3f} prec={b.best_recall_precision:.3f} f1={b.best_recall_f1:.3f}")
    print('\n--- best-F1 rollout document (truncated) ---')
    print(b.best_f1_document[:600], '...')

_ = explore(t.sort_values('best_f1').iloc[-1].entry_id)  # highest best-F1 target

### Interactive picker


In [ ]:
try:
    import ipywidgets as W
    order = t.sort_values('best_f1', ascending=False)
    opts = [(f'{r.entry_id}  (L={int(r.L)}, bestF1={r.best_f1:.2f})', r.entry_id) for r in order.itertuples()]
    W.interact(lambda entry: explore(entry), entry=W.Dropdown(options=opts, description='target'))
except Exception as e:
    print('ipywidgets unavailable — call explore("AF-...") directly:', e)